# 面向机器学习的微积分（Calculus for Machine Learning）

对应课程：`phases/01-math-foundations/04-calculus-for-ml`

> 导数告诉你哪边是下坡。神经网络学习所需的全部信息尽在于此。

本 notebook 把 `derivatives.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `derivatives.py`。

**贯穿全课的模式：** 导数给出下坡方向；梯度下降沿 $-\nabla f$ 走；Hessian 告诉你谷底还是鞍点。


## 0. 依赖

只用标准库。


In [1]:
import math


## 1. 数值导数（中心差分）

$$
f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}
$$

比单侧差分少一阶误差。$f(x)=x^2$ 在 $x=3$ 处解析导数是 $6$。


In [2]:
def numerical_derivative(f, x, h=1e-7):
    """中心差分。h 太小会抵消，太大有截断误差。"""
    return (f(x + h) - f(x - h)) / (2 * h)


f = lambda x: x ** 2
approx = numerical_derivative(f, 3.0)
print("d/dx x^2 |_{x=3} ≈", approx)
print("解析值 6, 误差", abs(approx - 6.0))


d/dx x^2 |_{x=3} ≈ 5.999999990180527
解析值 6, 误差 9.819473234529141e-09


## 2. 数值梯度

$$
\nabla f(\mathbf{x})_i \approx \frac{f(\mathbf{x}+h\mathbf{e}_i)-f(\mathbf{x}-h\mathbf{e}_i)}{2h}
$$

一次只扰动一个坐标。$f(x,y)=x^2+y^2$ 在 `[1,2]` 的梯度是 `[2,4]`。


In [3]:
def numerical_gradient(f, point, h=1e-7):
    """对每个分量做中心差分，拼成梯度向量。"""
    gradient = []
    for i in range(len(point)):
        point_plus = list(point)
        point_minus = list(point)
        point_plus[i] += h
        point_minus[i] -= h
        partial = (f(point_plus) - f(point_minus)) / (2 * h)
        gradient.append(partial)
    return gradient


def bowl(point):
    x, y = point
    return x ** 2 + y ** 2


g = numerical_gradient(bowl, [1.0, 2.0])
print("∇(x²+y²) at [1,2] ≈", [round(v, 8) for v in g])


∇(x²+y²) at [1,2] ≈ [2.0, 4.0]


## 3. 一维梯度下降

$$
x_{t+1} = x_t - \eta\, f'(x_t)
$$

最小化 $f(x)=(x-3)^2$，从 $x=0$ 出发应走向 $3$。解析导数 $f'=2(x-3)$。


In [4]:
def gradient_descent_1d(f, df, x0, lr=0.1, steps=20):
    """沿 -f' 走。返回终点和 (step, x, f(x)) 历史。"""
    x = x0
    history = []
    for step in range(steps):
        grad = df(x)
        x = x - lr * grad
        history.append((step, x, f(x)))
    return x, history


f = lambda x: (x - 3) ** 2
df = lambda x: 2 * (x - 3)
xmin, hist = gradient_descent_1d(f, df, x0=0.0, lr=0.3, steps=15)
print("起点 x=0, f=9")
for step, x, fx in hist:
    if step % 4 == 0 or step == 14:
        print(f"  step {step:2d}  x={x:.6f}  f={fx:.6f}")
print("终点 x =", round(xmin, 6), "（真值 3）")


起点 x=0, f=9
  step  0  x=1.800000  f=1.440000
  step  4  x=2.969280  f=0.000944
  step  8  x=2.999214  f=0.000001
  step 12  x=2.999980  f=0.000000
  step 14  x=2.999997  f=0.000000
终点 x = 2.999997 （真值 3）


## 4. 多维梯度下降

$$
\mathbf{x}_{t+1} = \mathbf{x}_t - \eta\,\nabla f(\mathbf{x}_t)
$$

没有解析梯度时，用上一节的数值梯度。碗 $x^2+y^2$ 从 `[4,3]` 走向原点。


In [5]:
def gradient_descent_nd(f, x0, lr=0.1, steps=100):
    """每步用 numerical_gradient，沿 -∇f 更新。"""
    point = list(x0)
    history = []
    for step in range(steps):
        grad = numerical_gradient(f, point)
        point = [p - lr * g for p, g in zip(point, grad)]
        history.append((step, list(point), f(point)))
    return point, history


end, hist = gradient_descent_nd(bowl, [4.0, 3.0], lr=0.2, steps=20)
print("起点 [4, 3]")
for step, p, fx in hist:
    if step % 5 == 0 or step == 19:
        print(f"  step {step:2d}  ({p[0]:.4f}, {p[1]:.4f})  f={fx:.6f}")
print("终点", [round(x, 6) for x in end])


起点 [4, 3]
  step  0  (2.4000, 1.8000)  f=9.000000
  step  5  (0.1866, 0.1400)  f=0.054420
  step 10  (0.0145, 0.0109)  f=0.000329
  step 15  (0.0011, 0.0008)  f=0.000002
  step 19  (0.0001, 0.0001)  f=0.000000
终点 [0.000146, 0.00011]


## 5. Hessian

$$
H=\begin{bmatrix}f_{xx}&f_{xy}\\f_{yx}&f_{yy}\end{bmatrix}
$$

二阶偏导矩阵。$x^2+y^2$ 处处 $H=\mathrm{diag}(2,2)$：两个正特征值，局部极小（碗底）。


In [6]:
def hessian_2d(f, x, y, h=1e-5):
    """二阶中心差分。f 的签名是 f(x, y)，不是点列表。"""
    fxx = (f(x + h, y) - 2 * f(x, y) + f(x - h, y)) / (h ** 2)
    fyy = (f(x, y + h) - 2 * f(x, y) + f(x, y - h)) / (h ** 2)
    fxy = (
        f(x + h, y + h) - f(x + h, y - h) - f(x - h, y + h) + f(x - h, y - h)
    ) / (4 * h ** 2)
    return [[fxx, fxy], [fxy, fyy]]


def bowl_xy(x, y):
    return x ** 2 + y ** 2


H = hessian_2d(bowl_xy, 0.0, 0.0)
print("H(x²+y²) at (0,0):")
for row in H:
    print(" ", [round(v, 6) for v in row])


H(x²+y²) at (0,0):
  [2.0, 0.0]
  [0.0, 2.0]


## 6. Hessian 特征值：极小 vs 鞍点

$2\times 2$ 的 $\lambda$ 仍由 $\mathrm{tr}$、$\det$ 求出。两个正 → 局部极小；一正一负 → 鞍点。


In [7]:
def hessian_eigenvalues(H):
    """2x2 对称阵特征值。判别式为负则返回 (None, None)。"""
    a, b = H[0][0], H[0][1]
    c, d = H[1][0], H[1][1]
    trace = a + d
    det = a * d - b * c
    discriminant = trace ** 2 - 4 * det
    if discriminant < 0:
        return None, None
    sqrt_disc = discriminant ** 0.5
    return (trace + sqrt_disc) / 2, (trace - sqrt_disc) / 2


Hb = hessian_2d(bowl_xy, 0.0, 0.0)
print("碗 x²+y² 特征值:", tuple(round(v, 6) for v in hessian_eigenvalues(Hb)))

def saddle(x, y):
    return x ** 2 - y ** 2

Hs = hessian_2d(saddle, 0.0, 0.0)
print("鞍 x²-y² 特征值:", tuple(round(v, 6) for v in hessian_eigenvalues(Hs)))


碗 x²+y² 特征值: (2.0, 2.0)
鞍 x²-y² 特征值: (2.0, -2.0)


## 7. 泰勒展开

$$
f(x_0+h)\approx f(x_0)+f'(x_0)h+\tfrac12 f''(x_0)h^2
$$

在 $x_0=0$ 处 $e^x$ 的 0/1/2 阶分别是 $1$、$1+h$、$1+h+h^2/2$。$h$ 越小越准。


In [8]:
def taylor_approx(f, f_prime, f_double_prime, x0, h, order=2):
    """0/1/2 阶泰勒。e^x 的各阶导数仍是 e^x。"""
    result = f(x0)
    if order >= 1:
        result += f_prime(x0) * h
    if order >= 2:
        result += 0.5 * f_double_prime(x0) * h ** 2
    return result


x0 = 0.0
print(f"{'h':>6}  {'true e^h':>12}  {'ord0':>10}  {'ord1':>10}  {'ord2':>10}")
for h in [0.1, 0.5, 1.0]:
    true_val = math.exp(x0 + h)
    t0 = taylor_approx(math.exp, math.exp, math.exp, x0, h, order=0)
    t1 = taylor_approx(math.exp, math.exp, math.exp, x0, h, order=1)
    t2 = taylor_approx(math.exp, math.exp, math.exp, x0, h, order=2)
    print(f"{h:6.1f}  {true_val:12.6f}  {t0:10.6f}  {t1:10.6f}  {t2:10.6f}")


     h      true e^h        ord0        ord1        ord2
   0.1      1.105171    1.000000    1.100000    1.105000
   0.5      1.648721    1.000000    1.500000    1.625000
   1.0      2.718282    1.000000    2.000000    2.500000


## 对照表

| 函数 | 角色 |
|------|------|
| `numerical_derivative` | 一维中心差分 |
| `numerical_gradient` | 逐坐标扰动，拼成 $\nabla f$ |
| `gradient_descent_1d` | $x\leftarrow x-\eta f'(x)$ |
| `gradient_descent_nd` | $\mathbf{x}\leftarrow\mathbf{x}-\eta\nabla f$ |
| `hessian_2d` | 二阶差分，曲率矩阵 |
| `hessian_eigenvalues` | 正定=谷底，变号=鞍点 |
| `taylor_approx` | 用导数在附近冒充 $f$ |

要看完整打印 demo，运行：

```bash
python derivatives.py
```
